In [ ]:
import pandas as pd
import xarray as xr
import pandas as pd
import os
import time

print("done")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os

dm = pd.read_csv("data/Dataset_Maestro_Amplificado_ML.csv")

dm.info()

#guardar en excel dm.info

#imprimir la cantidad de datos con parasitos por centro

dm.drop

#print




In [ ]:
import pandas as pd

ruta = "data/Dataset_Maestro_Global_ML.csv"
df = pd.read_csv(ruta)
df['id_centro'] = df['id_centro'].astype(str)

variables_biologicas = ['Hembras_Ovigeras', 'Adultos_Moviles', 'Juveniles']

# Una fila tiene datos de parásitos si TODAS las variables biológicas no son NaN
con_parasitos = df.dropna(subset=variables_biologicas)

# Conteo por centro
conteo = con_parasitos.groupby('id_centro').size().rename('Registros_con_Parasitos')

# Contexto: total de filas por centro y % de cobertura
total = df.groupby('id_centro').size().rename('Total_Registros')
resumen = pd.concat([total, conteo], axis=1).fillna(0).astype(int)
resumen['%_Cobertura'] = (resumen['Registros_con_Parasitos'] / resumen['Total_Registros'] * 100).round(1)

print(resumen)
print(f"\nTotal de registros con parásitos: {resumen['Registros_con_Parasitos'].sum()}")

In [ ]:
import pandas as pd
import os

# ==========================================
# 1. Configuración de Rutas y Diccionarios
# ==========================================
ruta_parasitos_global = "data/Cargas_parasitarias_2012_2024.xlsx"
directorio_salida_global = "data"

centros = {
    "110557": "coleccion/Historico_110557/Consolidado_Ambiental_Centro_110557.csv",
    "120069": "coleccion/Historico_120069/Consolidado_Ambiental_Centro_120069.csv",
    "110745": "coleccion/Historico_110745/Consolidado_Ambiental_Centro_110745.csv"
}

print("Iniciando integración relacional de matrices ambientales y epidemiológicas...")

# ==========================================
# 2. Procesamiento del Dataset Epidemiológico Global
# ==========================================
df_par_global = pd.read_excel(ruta_parasitos_global)

# Mapeo estricto basado en los encabezados reales del archivo fuente
df_par_global = df_par_global.rename(columns={
    'Código Centro': 'id_centro',
    'Año': 'Año_ISO',              
    'semana': 'Semana',            
    'Prom. Hembras Ovígeras': 'Hembras_Ovigeras',
    'Prom. Adultos Móviles': 'Adultos_Moviles',
    'Prom. Juveniles': 'Juveniles'
})

# Estandarización de tipos de datos para la clave primaria
df_par_global['id_centro'] = df_par_global['id_centro'].astype(str)

columnas_parasitos = ['Hembras_Ovigeras', 'Adultos_Moviles', 'Juveniles']

# Reducción de duplicados mediante agrupamiento por clave temporal y espacial
df_par_global = df_par_global.groupby(['id_centro', 'Año_ISO', 'Semana'])[columnas_parasitos].mean().reset_index()

# ==========================================
# 3. Procesamiento y Apilamiento de Datos Ambientales
# ==========================================
lista_ambiental = []

renombre_ambiental = {
    'time': 'Fecha',
    'thetao': 'Temperatura_C',
    'so': 'Salinidad_psu',
    'uo': 'Corriente_E_m_s',
    'vo': 'Corriente_N_m_s',
    'zos': 'Altura_Mar_m',
    'mlotst': 'Capa_Mezcla_m'
}

for id_centro, ruta_amb in centros.items():
    if not os.path.exists(ruta_amb):
        print(f"[ERROR] Archivo ambiental ausente para el centro {id_centro}.")
        continue
        
    df_amb = pd.read_csv(ruta_amb, parse_dates=['Fecha'])
    
    # Estandarización de columnas físicas
    df_amb = df_amb.rename(columns=renombre_ambiental)
    
    # Ingeniería de características temporales
    df_amb['Año'] = df_amb['Fecha'].dt.year
    df_amb['Mes'] = df_amb['Fecha'].dt.month
    df_amb['Dia'] = df_amb['Fecha'].dt.day
    df_amb['Semana'] = df_amb['Fecha'].dt.isocalendar().week
    df_amb['Año_ISO'] = df_amb['Fecha'].dt.isocalendar().year
    
    # Inyección de llave foránea espacial
    df_amb.insert(0, 'id_centro', str(id_centro))
    
    lista_ambiental.append(df_amb)
    print(f"[OK] Matriz ambiental procesada: Centro {id_centro}")

# Consolidación vertical
df_ambiental_total = pd.concat(lista_ambiental, ignore_index=True)

# ==========================================
# 4. Cruce Relacional (Data Merge)
# ==========================================
print("\nEjecutando cruce relacional maestro (LEFT JOIN)...")
df_maestro_total = pd.merge(
    df_ambiental_total, 
    df_par_global, 
    on=['id_centro', 'Año_ISO', 'Semana'], 
    how='left'
)


df_maestro_total = df_maestro_total.drop(columns=['Año_ISO'])

# ==========================================
# 5. Ordenamiento y Exportación
# ==========================================
columnas_deseadas = [
    'id_centro', 'Fecha', 'Año', 'Mes', 'Dia', 'Semana', 
    'Temperatura_C', 'Salinidad_psu', 'Corriente_E_m_s', 'Corriente_N_m_s', 
    'Altura_Mar_m', 'Capa_Mezcla_m', 
    'Hembras_Ovigeras', 'Adultos_Moviles', 'Juveniles'
]


columnas_existentes = [c for c in columnas_deseadas if c in df_maestro_total.columns]
df_maestro_total = df_maestro_total[columnas_existentes]

ruta_salida = os.path.join(directorio_salida_global, "Dataset_Maestro_Global3.csv")

df_maestro_total.to_csv(ruta_salida, index=False, encoding='utf-8', na_rep='N/A')

print(f"\nOperación finalizada. Dataset maestro exportado en: {ruta_salida}")
print(f"Dimensiones de la matriz: {df_maestro_total.shape}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os

# ==========================================
# 1. Configuración de Entorno
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Global_Revision.csv" 
directorio_salida = "data/Graficos_Comparativos"
os.makedirs(directorio_salida, exist_ok=True)


sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 300 

print("Iniciando generación de gráficos comparativos...")

# ==========================================
# 2. Carga y Preparación de Datos
# ==========================================
df = pd.read_csv(ruta_dataset, parse_dates=['Fecha'])


df['id_centro'] = df['id_centro'].astype(str)


variables_excluidas = ['id_centro', 'Fecha', 'Año', 'Mes', 'Dia', 'Semana']
variables_a_graficar = [col for col in df.columns if col not in variables_excluidas]

# ==========================================
# 3. Ciclo de Renderizado de Gráficos
# ==========================================
for variable in variables_a_graficar:
    print(f"-> Generando gráfico para: {variable}")
    
   
    fig, ax = plt.subplots(figsize=(14, 6))
    
   
    sns.lineplot(
        data=df, 
        x='Fecha', 
        y=variable, 
        hue='id_centro',
        palette='Set1', 
        linewidth=1.2,
        alpha=0.85,
        ax=ax
    )
    
    
    titulo_limpio = variable.replace("_", " ")
    ax.set_title(f"Evolución Temporal: {titulo_limpio} (2020 - 2025)", fontsize=16, fontweight='bold', pad=15)
    ax.set_ylabel(titulo_limpio, fontweight='bold')
    ax.set_xlabel('Año', fontweight='bold')
    
    # ==========================================
    # 4. Formateo del Eje Temporal
    # ==========================================

    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    

    ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_minor_formatter(mdates.DateFormatter('%b'))
    

    plt.setp(ax.xaxis.get_majorticklabels(), fontweight='bold', fontsize=11)
    plt.setp(ax.xaxis.get_minorticklabels(), rotation=45, fontsize=9, color='gray')
    

    ax.legend(title='Centro de Cultivo', title_fontsize='11', fontsize='10', loc='upper left', bbox_to_anchor=(1, 1))
    

    plt.tight_layout()
    ruta_imagen = os.path.join(directorio_salida, f"Comparativa_{variable}.png")
    plt.savefig(ruta_imagen, bbox_inches='tight')
    plt.close()

print(f"\n[ÉXITO] Todos los gráficos han sido guardados en la carpeta: {directorio_salida}")

In [ ]:
import pandas as pd
import os

# ==========================================
# 1. Configuración de Rutas
# ==========================================
ruta_origen = "data/Cargas_parasitarias_2012_2024.xlsx" 
ruta_destino = "data/Datos_Parasitarios_Magallanes_Aysen.xlsx"


centros_a_extraer = ['110557', '120069', '110745'] 

# ==========================================
# 2. Ingesta de Datos
# ==========================================
print("Ingestando matriz sanitaria original...")
df = pd.read_excel(ruta_origen)

# ==========================================
# 3. Estandarización de Encabezados (Parsing)
# ==========================================

df = df.rename(columns={
    'Código Centro': 'id_centro',
    'Año': 'Año_ISO',              
    'semana': 'Semana',            
    'Prom. Hembras Ovígeras': 'Hembras_Ovigeras',
    'Prom. Adultos Móviles': 'Adultos_Moviles',
    'Prom. Juveniles': 'Juveniles',
    'Salinidad': 'Salinidad',
    'Temperatura': 'Temperatura'
})

# ==========================================
# 4. Limpieza de Formatos
# ==========================================

df['id_centro'] = df['id_centro'].astype(str)


if 'fecha' in df.columns:
    df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce').dt.date

# ==========================================
# 5. Filtrado y Clonación
# ==========================================
print(f"Filtrando registros para los centros: {centros_a_extraer}...")


df_extraido = df[df['id_centro'].isin(centros_a_extraer)].copy()

# ==========================================
# 6. Exportación Segura
# ==========================================

os.makedirs(os.path.dirname(ruta_destino), exist_ok=True)

print(f"Se aislaron {len(df_extraido)} registros.")
print("Escribiendo nuevo archivo Excel...")

df_extraido.to_excel(ruta_destino, index=False, engine='openpyxl')

print(f"[ÉXITO] Matriz sanitaria estandarizada y exportada en: {ruta_destino}")

In [ ]:
# ==========================================
# PIPELINE ETL: FUSIÓN AMBIENTAL - EPIDEMIOLÓGICA
# ==========================================
import pandas as pd
import os

# ==========================================
# 1. Configuración de Rutas y Diccionarios
# ==========================================
ruta_parasitos_global = "data/Cargas_parasitarias_2012_2024.xlsx"
directorio_salida_global = "data" 

centros = {
    "110557": "coleccion/Historico_110557/Consolidado_Ambiental_Centro_110557.csv",
    "120069": "coleccion/Historico_120069/Consolidado_Ambiental_Centro_120069.csv",
    "110745": "coleccion/Historico_110745/Consolidado_Ambiental_Centro_110745.csv"
}

print("Iniciando integración relacional de matrices ambientales y epidemiológicas...")

# ==========================================
# 2. Procesamiento del Dataset Epidemiológico Global
# ==========================================
df_par_global = pd.read_excel(ruta_parasitos_global)

df_par_global = df_par_global.rename(columns={
    'Código Centro': 'id_centro',
    'Año': 'Año_ISO',              
    'semana': 'Semana',            
    'Prom. Hembras Ovígeras': 'Hembras_Ovigeras',
    'Prom. Adultos Móviles': 'Adultos_Moviles',
    'Prom. Juveniles': 'Juveniles'
})


df_par_global['id_centro'] = df_par_global['id_centro'].astype(str)

columnas_parasitos = ['Hembras_Ovigeras', 'Adultos_Moviles', 'Juveniles']


df_par_global = df_par_global.groupby(['id_centro', 'Año_ISO', 'Semana'])[columnas_parasitos].mean().reset_index()

# ==========================================
# 3. Procesamiento y Apilamiento de Datos Ambientales
# ==========================================
lista_ambiental = []

renombre_ambiental = {
    'time': 'Fecha',
    'thetao': 'Temperatura_C',
    'so': 'Salinidad_psu',
    'uo': 'Corriente_E_m_s',
    'vo': 'Corriente_N_m_s',
    'zos': 'Altura_Mar_m',
    'mlotst': 'Capa_Mezcla_m'
}

for id_centro, ruta_amb in centros.items():
    if not os.path.exists(ruta_amb):
        print(f"[ERROR] Archivo ambiental ausente para el centro {id_centro}.")
        continue
        
    df_amb = pd.read_csv(ruta_amb, parse_dates=['Fecha'])
    
    df_amb = df_amb.rename(columns=renombre_ambiental)
    

    df_amb['Año'] = df_amb['Fecha'].dt.year
    df_amb['Mes'] = df_amb['Fecha'].dt.month
    df_amb['Dia'] = df_amb['Fecha'].dt.day
    df_amb['Semana'] = df_amb['Fecha'].dt.isocalendar().week

    df_amb['Año_ISO'] = df_amb['Fecha'].dt.isocalendar().year
    

    df_amb.insert(0, 'id_centro', str(id_centro))
    
    lista_ambiental.append(df_amb)
    print(f"[OK] Matriz ambiental procesada: Centro {id_centro}")


df_ambiental_total = pd.concat(lista_ambiental, ignore_index=True)

# ==========================================
# 4. Cruce Relacional (Data Merge)
# ==========================================
print("\nEjecutando cruce relacional maestro (LEFT JOIN)...")
df_maestro_total = pd.merge(
    df_ambiental_total, 
    df_par_global, 
    on=['id_centro', 'Año_ISO', 'Semana'], 
    how='left'
)


df_maestro_total = df_maestro_total.drop(columns=['Año_ISO'])

# ==========================================
# 5. Ordenamiento y Exportación
# ==========================================

columnas_deseadas = [
    'id_centro', 'Fecha', 'Año', 'Mes', 'Dia', 'Semana', 
    'Temperatura_C', 'Salinidad_psu', 'Corriente_E_m_s', 'Corriente_N_m_s', 
    'Altura_Mar_m', 'Capa_Mezcla_m', 
    'Hembras_Ovigeras', 'Adultos_Moviles', 'Juveniles'
]


columnas_existentes = [c for c in columnas_deseadas if c in df_maestro_total.columns]
df_maestro_total = df_maestro_total[columnas_existentes]


ruta_salida_ml = os.path.join(directorio_salida_global, "Dataset_Maestro_Global_ML.csv")
ruta_salida_excel = os.path.join(directorio_salida_global, "Dataset_Maestro_Global_Revision.csv")


df_maestro_total.to_csv(ruta_salida_ml, index=False, encoding='utf-8')

df_maestro_total.to_csv(ruta_salida_excel, index=False, encoding='utf-8', na_rep='N/A')

print("\nOperación de exportación dual finalizada.")
print(f"-> Insumo Modelo Predictivo (ML): {ruta_salida_ml}")
print(f"-> Insumo Auditoría (Excel): {ruta_salida_excel}")
print(f"Dimensiones de la matriz procesada: {df_maestro_total.shape}")

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Configuración de Rutas
# ==========================================
ruta_excel = "data/Datos_Parasitarios_Magallanes_Aysen.xlsx"
ruta_master = "data/Dataset_Maestro_Global_ML.csv"

# ==========================================
# 2. Ingesta y Tipificación
# ==========================================
print("Cargando matrices para auditoría estructural...")
df_excel = pd.read_excel(ruta_excel)
df_master = pd.read_csv(ruta_master)

df_excel['id_centro'] = df_excel['id_centro'].astype(str)
df_master['id_centro'] = df_master['id_centro'].astype(str)


df_master['Fecha'] = pd.to_datetime(df_master['Fecha'])

# ==========================================
# 3. Reconstrucción Temporal del "Puente" (Año ISO)
# ==========================================

df_master['Año_ISO_Calculado'] = df_master['Fecha'].dt.isocalendar().year

# ==========================================
# 4. Reducción Dimensional y Agregación
# ==========================================
columnas_parasitos = ['Hembras_Ovigeras', 'Adultos_Moviles', 'Juveniles']

# A. Procesamiento del Origen (Excel)
df_excel_semanal = df_excel.groupby(
    ['id_centro', 'Año_ISO', 'Semana'], as_index=False
)[columnas_parasitos].mean()

# B. Procesamiento del Dataset Maestro
df_master_validos = df_master.dropna(subset=['Hembras_Ovigeras']).copy()

# Agrupamos usando el Año ISO que acabamos de calcular
df_master_semanal = df_master_validos.groupby(
    ['id_centro', 'Año_ISO_Calculado', 'Semana'], as_index=False
)[columnas_parasitos].mean()


df_master_semanal = df_master_semanal.rename(columns={'Año_ISO_Calculado': 'Año_ISO'})

# ==========================================
# 5. Fusión Relacional para Validación
# ==========================================
print("Ejecutando cruce de auditoría por llaves [Centro, Año_ISO, Semana]...")
df_comparacion = pd.merge(
    df_excel_semanal,
    df_master_semanal,
    on=['id_centro', 'Año_ISO', 'Semana'],
    suffixes=('_Origen', '_Master')
)

print(f"Se consolidaron {len(df_comparacion)} bloques semanales para auditar.\n")

# ==========================================
# 6. Cálculo de Discrepancias
# ==========================================
errores_totales = 0

for col in columnas_parasitos:
    col_origen = f"{col}_Origen"
    col_master = f"{col}_Master"
    
    son_iguales = np.isclose(
        df_comparacion[col_origen].fillna(-999), 
        df_comparacion[col_master].fillna(-999), 
        atol=1e-5
    )
    
    cantidad_errores = len(son_iguales) - son_iguales.sum()
    errores_totales += cantidad_errores
    
    if cantidad_errores == 0:
        print(f" {col}: INTEGRIDAD CONFIRMADA. Desviación cero.")
    else:
        print(f" {col}: ANOMALÍA DETECTADA. {cantidad_errores} discrepancias.")
        df_errores = df_comparacion[~son_iguales]
        print(df_errores[['id_centro', 'Año_ISO', 'Semana', col_origen, col_master]].head())

# ==========================================
# 7. Veredicto Final
# ==========================================
print("-" * 60)
if errores_totales == 0:
    print(" AUDITORÍA SUPERADA: Los datos biológicos de tu Dataset Maestro coinciden al 100% con la base extraída de Sernapesca.")
else:
    print(" AUDITORÍA FALLIDA: Existen discrepancias. Revisar el método de agregación.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import os

# ==========================================
# 1. Configuración de Entorno y Rutas
# ==========================================
ruta_excel_insitu = "data/Cargas_parasitarias_2012_2024.xlsx"
ruta_csv_satelite = "data/Dataset_Maestro_Global_ML.csv"
ruta_grafico_salida = "data/Validacion_Ambiental_Por_Centro.png"

col_temp_insitu = 'Temperatura'
col_sal_insitu = 'Salinidad'
col_temp_sat = 'Temperatura_C'
col_sal_sat = 'Salinidad_psu'

print("Iniciando validación ambiental desglosada por centro...")

# ==========================================
# 2. Ingesta y Purificación de Tipos
# ==========================================
df_insitu = pd.read_excel(ruta_excel_insitu)
df_sat = pd.read_csv(ruta_csv_satelite)

df_insitu['id_centro'] = df_insitu['Código Centro'].astype(str)
df_sat['id_centro'] = df_sat['id_centro'].astype(str)
df_sat['Fecha'] = pd.to_datetime(df_sat['Fecha'])

# Forzar numéricos (Elimina los "N/A" convirtiéndolos en nulos reales)
df_insitu[col_temp_insitu] = pd.to_numeric(df_insitu[col_temp_insitu], errors='coerce')
df_insitu[col_sal_insitu] = pd.to_numeric(df_insitu[col_sal_insitu], errors='coerce')
df_sat[col_temp_sat] = pd.to_numeric(df_sat[col_temp_sat], errors='coerce')
df_sat[col_sal_sat] = pd.to_numeric(df_sat[col_sal_sat], errors='coerce')

# ==========================================
# 3. Preparación de Matrices (In Situ y Satélite)
# ==========================================
df_insitu = df_insitu.rename(columns={'Año': 'Año_ISO', 'semana': 'Semana'})
df_insitu_agrupado = df_insitu.groupby(
    ['id_centro', 'Año_ISO', 'Semana'], as_index=False
)[[col_temp_insitu, col_sal_insitu]].mean()

df_sat['Año_ISO'] = df_sat['Fecha'].dt.isocalendar().year
df_sat_agrupado = df_sat.groupby(
    ['id_centro', 'Año_ISO', 'Semana'], as_index=False
)[[col_temp_sat, col_sal_sat]].mean()

# ==========================================
# 4. Cruce Relacional
# ==========================================
df_validacion = pd.merge(
    df_insitu_agrupado,
    df_sat_agrupado,
    on=['id_centro', 'Año_ISO', 'Semana'],
    how='inner'
)

if df_validacion.empty:
    raise ValueError("No se encontraron coincidencias para realizar la validación.")

centros_unicos = df_validacion['id_centro'].unique()

# ==========================================
# 5. Reporte Estadístico y Gráficos
# ==========================================
sns.set_theme(style="whitegrid")

paleta_colores = sns.color_palette("Set1", n_colors=len(centros_unicos))

fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=300)

variables = [
    {'nombre': 'Temperatura', 'col_insitu': col_temp_insitu, 'col_sat': col_temp_sat, 'unidad': '°C', 'ax': axes[0]},
    {'nombre': 'Salinidad', 'col_insitu': col_sal_insitu, 'col_sat': col_sal_sat, 'unidad': 'psu', 'ax': axes[1]}
]

print("\n" + "=" * 65)
print("REPORTE ESTADÍSTICO DE VALIDACIÓN DESGLOSADO POR CENTRO")
print("=" * 65)

for var in variables:
    print(f"\n>>>> ANÁLISIS DE {var['nombre'].upper()} <<<<")
    

    df_var_global = df_validacion.dropna(subset=[var['col_insitu'], var['col_sat']])
    
    if df_var_global.empty:
        print(f"No hay datos suficientes para {var['nombre']}.")
        continue


    for i, centro in enumerate(centros_unicos):
        df_centro = df_var_global[df_var_global['id_centro'] == centro]
        
        if len(df_centro) < 2:
            print(f"\n--- Centro {centro}: Datos insuficientes (n={len(df_centro)}) ---")
            continue
            
        val_in = df_centro[var['col_insitu']]
        val_sat = df_centro[var['col_sat']]
        
        r_pearson, _ = pearsonr(val_in, val_sat)
        mae = np.mean(np.abs(val_in - val_sat))
        rmse = np.sqrt(np.mean((val_in - val_sat)**2))
        bias = np.mean(val_sat - val_in)
        
        var_in = np.var(val_in, ddof=1)
        var_sat = np.var(val_sat, ddof=1)
        std_in = np.std(val_in, ddof=1)
        std_sat = np.std(val_sat, ddof=1)
        
        print(f"\n--- Centro {centro} (n={len(df_centro)} mediciones coincidentes) ---")
        print(f"  Precisión del Modelo:")
        print(f"    - Correlación (r) : {r_pearson:.4f}")
        print(f"    - RMSE            : {rmse:.4f} {var['unidad']}")
        print(f"    - MAE             : {mae:.4f} {var['unidad']}")
        print(f"    - Sesgo (Bias)    : {bias:.4f} {var['unidad']}")
        print(f"  Métricas de Dispersión:")
        print(f"    - Varianza (σ²)   : In Situ = {var_in:.4f} | Satélite = {var_sat:.4f}")
        print(f"    - Desv. Est. (σ)  : In Situ = {std_in:.4f} | Satélite = {std_sat:.4f}")

    sns.scatterplot(
        data=df_var_global,
        x=var['col_sat'],
        y=var['col_insitu'],
        hue='id_centro',
        palette=paleta_colores,
        alpha=0.7,
        edgecolor='k',
        ax=var['ax']
    )
    
  
    min_val = min(df_var_global[var['col_insitu']].min(), df_var_global[var['col_sat']].min())
    max_val = max(df_var_global[var['col_insitu']].max(), df_var_global[var['col_sat']].max())
    
    var['ax'].plot(
        [min_val, max_val], [min_val, max_val], 
        color='black', linestyle='--', linewidth=1.5, label='Identidad (1:1)'
    )
    
   
    var['ax'].set_title(f"Dispersión de {var['nombre']} por Centro", fontsize=14, fontweight='bold', pad=10)
    var['ax'].set_xlabel(f"{var['nombre']} Satelital Copernicus ({var['unidad']})", fontweight='bold')
    var['ax'].set_ylabel(f"{var['nombre']} In Situ Sernapesca ({var['unidad']})", fontweight='bold')
    var['ax'].legend(title='ID Centro', loc='best')


plt.suptitle('Validación Oceanográfica Espacial: Satélite Copernicus vs In Situ', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ruta_grafico_salida, bbox_inches='tight')
plt.close()

print("\n" + "=" * 65)
print(f"[ÉXITO] Matriz de dispersión guardada en: {ruta_grafico_salida}")

In [ ]:
import pandas as pd

ruta_csv_satelite = "data/Dataset_Maestro_Global_ML.csv"

print("Cargando Dataset Maestro para análisis de valores nulos...")
df = pd.read_csv(ruta_csv_satelite)
df.info()
